# Step 3: Federated Learning Training
**TSU NSF AI Workshop 2026 — Federated Learning Lab**

---

## What are we doing in this notebook?

This is the heart of the lab. We will train an AI model across **2 simulated hospitals** using the **FedAvg** algorithm — without ever sharing patient data.

### The FedAvg Algorithm (McMahan et al., 2017)

```
SERVER starts with a random global model

For each communication round:
  ┌─────────────────────────────────────────────────┐
  │  1. SERVER sends global model to all clients    │
  │                                                 │
  │  2. CLIENT 1 trains on its local data           │
  │     CLIENT 2 trains on its local data           │
  │     (each client works independently)           │
  │                                                 │
  │  3. Clients send their model WEIGHTS to server  │
  │     (NOT the patient data — only numbers!)      │
  │                                                 │
  │  4. SERVER averages all weights → new model     │
  └─────────────────────────────────────────────────┘

Repeat for multiple rounds → model improves each round
```

**Privacy guarantee:** Raw images never leave the client.
Only model weights (just numbers, not images) are transmitted.

## 🔧 Mount Google Drive

We need the client data files created in notebook 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE  = "/content/drive/MyDrive/FederatedLearning"
DATA_DIR    = os.path.join(DRIVE_BASE, "data")
RESULTS_DIR = os.path.join(DRIVE_BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Check required files exist
for fname in ["client1.npz", "client2.npz", "test_data.npz"]:
    path = os.path.join(DATA_DIR, fname)
    status = "✅ Found" if os.path.exists(path) else "❌ Missing — run notebook 2 first!"
    print(f"  {fname}: {status}")

## Setup: Imports and Settings

Key settings you can experiment with:
- **NUM_ROUNDS** — how many times clients and server communicate
- **LOCAL_EPOCHS** — how much each client trains before sending weights back

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import json

# ---------- Federated Learning Settings ----------
NUM_ROUNDS    = 5     # How many communication rounds
LOCAL_EPOCHS  = 2     # Local training epochs per round per client
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
NUM_CLASSES   = 7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Settings: {NUM_ROUNDS} rounds, {LOCAL_EPOCHS} local epochs/round")

## Define the Model and Helper Functions

Same CNN architecture as in notebook 1 — all clients and the server use the same structure.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


def make_loader(images, labels, shuffle=True):
    x = torch.tensor(images / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
    y = torch.tensor(labels, dtype=torch.long)
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=shuffle)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(1) == lbls).sum().item()
            total   += lbls.size(0)
    return correct / total


print("Model and helpers defined.")

## The Two Core FL Functions

### `client_update` — what each hospital does
- Receives the global model from the server
- Trains it on **local data only**
- Returns the **updated weights** (not the data!)

### `federated_average` — what the server does
- Receives weight updates from all clients
- Computes a **weighted average** (clients with more data have more influence)
- This is the **FedAvg** algorithm

In [ ]:
def client_update(global_model, loader, local_epochs, lr):
    """
    CLIENT SIDE: Train on local data, return updated weights.
    In a real FL system this runs on the client's own machine.
    Raw data never leaves this function.
    """
    local_model = copy.deepcopy(global_model)  # Start from global model
    local_model.train()
    optimizer = optim.Adam(local_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(local_epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(local_model(imgs), lbls)
            loss.backward()
            optimizer.step()

    return local_model.state_dict()  # Return weights only (not data!)


def federated_average(global_model, client_weights, client_sizes):
    """
    SERVER SIDE: Compute weighted average of all client model weights.
    This is the FedAvg algorithm (McMahan et al., 2017).
    """
    total = sum(client_sizes)
    avg   = copy.deepcopy(client_weights[0])
    for key in avg:
        avg[key] = torch.zeros_like(avg[key], dtype=torch.float32)
    for weights, n in zip(client_weights, client_sizes):
        for key in avg:
            avg[key] += weights[key].float() * (n / total)  # Weighted sum
    global_model.load_state_dict(avg)
    return global_model


print("FL functions defined.")

## Load Data

Each client loads only their own data. In a real system, each hospital would only ever have access to their own files.

In [ ]:
c1 = np.load(os.path.join(DATA_DIR, "client1.npz"))
c2 = np.load(os.path.join(DATA_DIR, "client2.npz"))
td = np.load(os.path.join(DATA_DIR, "test_data.npz"))

client1_loader = make_loader(c1["images"], c1["labels"])
client2_loader = make_loader(c2["images"], c2["labels"])
test_loader    = make_loader(td["images"], td["labels"], shuffle=False)
client_sizes   = [len(c1["labels"]), len(c2["labels"])]

print(f"Client 1 (Health Center A): {client_sizes[0]} samples")
print(f"Client 2 (Health Center B): {client_sizes[1]} samples")
print(f"Test set (shared):          {len(td['labels'])} samples")

## Run Federated Training

Watch the accuracy increase round by round as the global model improves from aggregated knowledge.

Notice that **no client ever sees the other client's data** — they only share weights.

In [ ]:
print(f"Starting Federated Training: {NUM_ROUNDS} rounds, {LOCAL_EPOCHS} local epochs/round\n")

# Initialize global model on the server
global_model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
history = {"round": [], "test_acc": []}

for rnd in range(1, NUM_ROUNDS + 1):
    print(f"===== Round {rnd}/{NUM_ROUNDS} =====")

    # Step 1: Each client trains on local data
    print(f"  [Client 1] Training locally on {client_sizes[0]} samples...")
    w1 = client_update(global_model, client1_loader, LOCAL_EPOCHS, LEARNING_RATE)

    print(f"  [Client 2] Training locally on {client_sizes[1]} samples...")
    w2 = client_update(global_model, client2_loader, LOCAL_EPOCHS, LEARNING_RATE)

    # Step 2: Server aggregates (averages) the weights
    print(f"  [Server]   Aggregating weights with FedAvg...")
    global_model = federated_average(global_model, [w1, w2], client_sizes)

    # Step 3: Evaluate the new global model
    acc = evaluate(global_model, test_loader)
    print(f"  [Server]   Global model test accuracy: {acc:.2%}\n")

    history["round"].append(rnd)
    history["test_acc"].append(round(acc, 4))

print(f"Final Federated Test Accuracy: {history['test_acc'][-1]:.2%}")

## Save Results to Google Drive

In [ ]:
results = {
    "method"      : "Federated",
    "num_rounds"  : NUM_ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "test_acc"    : history["test_acc"][-1],
    "history"     : history,
}
path = os.path.join(RESULTS_DIR, "federated_results.json")
with open(path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {path}")
print("\n✅ Done! Open notebook 4_evaluation_visualization.ipynb next.")

## Summary

| Communication Round | What happened |
|--------------------|--------------|
| Each round | Client 1 & Client 2 trained locally, sent weights to server |
| Each round | Server averaged weights (FedAvg), updated global model |
| End | Global model evaluated on shared test set |

**What data was transmitted?** Only model weights (~2 MB per round per client).
**What stayed private?** All patient images — they never left each client.

➡️ **Next step: `4_evaluation_visualization.ipynb`** — compare centralized vs. federated with charts.